# Week3: Final HTML Parsing Engine

**Goal:**  
Finalize the HTML-to-DataFrame parsing engine supports both **colspan** and **rowspan** and implement error handling for misaligned cells.  

The pipeline now:
1. Parses HTML table structures.
2. Expands horizontally merged cells using `colspan`.
3. Carries vertically merged cells using `rowspan`.
4. Detects inconsistent row lengths.
5. Applies row normalization only when needed.
6. Records parsing status and error types for batch evaluation.

For cases such as **"Year Ended December 31, 2017"**, the parser preserves the original HTML text because the combined label already exists in the source HTML. Splitting it into multiple semantic header levels would require additional semantic post-processing rather than basic rowspan/colspan parsing.

---

## **Standard invocation template**  
You can directly use the `robust_html_to_dataframe()` function as the entry point. Just pass the model’s predicted HTML into the function to obtain a structured DataFrame, which can then be used for evaluation (e.g., TEDS). The lower-level parsing functions are internal and not intended for direct use.

In [ ]:
predicted_html = model_output_html  # model output

result = robust_html_to_dataframe(predicted_html)

if result["status"] != "failed":
    df_pred = result["df"]
else:
    print("Parsing failed:", result["error_type"])

If verification is to be conducted (compared with ground true)

In [ ]:
gt_html = record["html_table"]

df_gt = robust_html_to_dataframe(gt_html)["df"]
df_pred = robust_html_to_dataframe(predicted_html)["df"]

# Next step can do the TEDS/comparison

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

In [3]:
BASE_PATH = "/content/drive/MyDrive/cv project"

JSONL_PATH = os.path.join(
    BASE_PATH,
    "hierarchical_tables_v1",
    "data",
    "processed",
    "hard_examples_subset.jsonl"
)

CSV_PATH = os.path.join(BASE_PATH, "df_combined.csv")

OUTPUT_DIR = os.path.join(BASE_PATH, "sprint3_final_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("JSONL exists:", os.path.exists(JSONL_PATH))
print("CSV exists:", os.path.exists(CSV_PATH))
print("Output dir:", OUTPUT_DIR)

JSONL exists: True
CSV exists: False
Output dir: /content/drive/MyDrive/cv project/sprint3_final_outputs


### Load Original JSONL Records

In [4]:
records = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Total records:", len(records))
print(records[0].keys())

Total records: 32670
dict_keys(['html_table', 'imgid'])


### Load Week 1 Combined Feature Dataset

In [6]:

print(df_combined.shape)
df_combined.head()

(32670, 21)


,num_rows,num_cells,max_colspan,max_rowspan,has_colspan,has_rowspan,is_complex,hard_score,imgid,img_file,...,height,aspect_ratio,width_bin,height_bin,aspect_ratio_bin,blur_score,blur_bin,contrast_score,contrast_bin,image_type
0,9,34,3,1,True,False,True,12.5,fintabnet_000001,fintabnet_000001.png,...,464.0,3.127155,"(1200, 1500]","(300, 600]","(3, 5]",3678.917551,"(2000, 4000]",43.968108,"(40, 50]",wide_table
1,40,158,3,1,True,False,True,28.0,fintabnet_000004,fintabnet_000004.png,...,1826.0,0.795181,"(1200, 1500]","(1600, 2000]","(0, 1]",2894.365766,"(2000, 4000]",38.080415,"(20, 40]",tall_table
2,6,55,2,2,True,True,True,11.0,fintabnet_000006,fintabnet_000006.png,...,366.0,4.144809,"(1500, 2000]","(300, 600]","(3, 5]",6247.870952,"(6000, 8000]",55.573495,"(50, 60]",wide_table
3,7,26,3,1,True,False,True,11.5,fintabnet_000007,fintabnet_000007.png,...,330.0,4.015152,"(1200, 1500]","(300, 600]","(3, 5]",4598.369386,"(4000, 6000]",51.369766,"(50, 60]",wide_table
4,12,46,2,2,True,True,True,14.0,fintabnet_000008,fintabnet_000008.png,...,527.0,2.755218,"(1200, 1500]","(300, 600]","(2, 3]",7751.798653,"(6000, 8000]",38.023786,"(20, 40]",low_contrast


### Map HTML Back to df_combined by imgid

In [7]:
imgid_to_html = {r["imgid"]: r["html_table"] for r in records}

df_combined["html_table"] = df_combined["imgid"].map(imgid_to_html)

print("Missing HTML:", df_combined["html_table"].isna().sum())
df_combined[["imgid", "html_table"]].head()

Missing HTML: 0


,imgid,html_table
0,fintabnet_000001,<table>\n <tr>\n <td>\n </td>\n <td colspan...
1,fintabnet_000004,<table>\n <tr>\n <td>\n </td>\n <td colspan...
2,fintabnet_000006,<table>\n <tr>\n <td>\n </td>\n <td colspan...
3,fintabnet_000007,<table>\n <tr>\n <td>\n </td>\n <td colspan...
4,fintabnet_000008,<table>\n <tr>\n <td>\n </td>\n <td colspan...


### Create Complex Samples for Testing

In [8]:
complex_samples = df_combined[
    (df_combined["max_colspan"] > 1) &
    (df_combined["max_rowspan"] > 1)
].copy()

print("Complex samples:", len(complex_samples))

complex_samples[["imgid", "max_colspan", "max_rowspan"]].head()

Complex samples: 1510


,imgid,max_colspan,max_rowspan
2,fintabnet_000006,2,2
4,fintabnet_000008,2,2
7,fintabnet_000012,2,2
14,fintabnet_000026,2,2
15,fintabnet_000028,2,2


### Final Robust Parser

In [9]:
def parse_table_with_span(html):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")

    if table is None:
        raise ValueError("no_table_found")

    grid = []
    rowspan_map = {}

    for row_idx, tr in enumerate(table.find_all("tr")):
        row = []
        col_idx = 0

        for cell in tr.find_all(["td", "th"]):

            # Fill positions already occupied by previous rowspan cells
            while (row_idx, col_idx) in rowspan_map:
                value, remaining = rowspan_map.pop((row_idx, col_idx))
                row.append(value)

                if remaining > 1:
                    rowspan_map[(row_idx + 1, col_idx)] = (value, remaining - 1)

                col_idx += 1

            text = cell.get_text(" ", strip=True)

            try:
                colspan = int(cell.get("colspan", 1))
            except:
                colspan = 1

            try:
                rowspan = int(cell.get("rowspan", 1))
            except:
                rowspan = 1

            # Expand colspan horizontally
            for offset in range(colspan):
                row.append(text)

                # Carry value down if rowspan exists
                if rowspan > 1:
                    rowspan_map[(row_idx + 1, col_idx + offset)] = (text, rowspan - 1)

            col_idx += colspan

        # Fill remaining rowspan cells after the last normal cell
        while (row_idx, col_idx) in rowspan_map:
            value, remaining = rowspan_map.pop((row_idx, col_idx))
            row.append(value)

            if remaining > 1:
                rowspan_map[(row_idx + 1, col_idx)] = (value, remaining - 1)

            col_idx += 1

        grid.append(row)

    if len(grid) == 0:
        raise ValueError("empty_table")

    return grid

### Normalize Rows if Needed

In [10]:
def normalize_rows(rows, fill_value=""):
    max_len = max(len(row) for row in rows)
    return [row + [fill_value] * (max_len - len(row)) for row in rows]


def check_row_alignment(rows):
    row_lengths = [len(row) for row in rows]

    return {
        "min_cols": min(row_lengths),
        "max_cols": max(row_lengths),
        "unique_lengths": sorted(set(row_lengths)),
        "is_aligned": len(set(row_lengths)) == 1
    }

### Final HTML-to-DataFrame Function with Error Handling

In [11]:
def robust_html_to_dataframe(html):
    """
    Final Sprint 3 parser:
    - Handles colspan
    - Handles rowspan
    - Detects misaligned rows
    - Normalizes rows when needed
    - Returns DataFrame + parsing status
    """

    try:
        rows = parse_table_with_span(html)

        alignment_info = check_row_alignment(rows)

        if alignment_info["is_aligned"]:
            status = "success_clean"
            normalized = False
        else:
            rows = normalize_rows(rows)
            status = "success_fixed_misalignment"
            normalized = True

        df = pd.DataFrame(rows)

        result = {
            "status": status,
            "error_type": "",
            "error_message": "",
            "normalized": normalized,
            "min_cols_before": alignment_info["min_cols"],
            "max_cols_before": alignment_info["max_cols"],
            "unique_lengths_before": alignment_info["unique_lengths"],
            "num_rows": df.shape[0],
            "num_cols": df.shape[1],
            "df": df
        }

        return result

    except Exception as e:
        return {
            "status": "failed",
            "error_type": str(e),
            "error_message": str(e),
            "normalized": False,
            "min_cols_before": None,
            "max_cols_before": None,
            "unique_lengths_before": None,
            "num_rows": None,
            "num_cols": None,
            "df": None
        }

### Test One Complex Sample

In [12]:
sample = complex_samples.iloc[0]

html = sample["html_table"]
imgid = sample["imgid"]

print("imgid:", imgid)
print("max_colspan:", sample["max_colspan"])
print("max_rowspan:", sample["max_rowspan"])

result = robust_html_to_dataframe(html)

print("status:", result["status"])
print("normalized:", result["normalized"])
print("shape:", result["num_rows"], result["num_cols"])
print("unique row lengths before:", result["unique_lengths_before"])

display(result["df"])

imgid: fintabnet_000006
max_colspan: 2
max_rowspan: 2
status: success_clean
normalized: False
shape: 6 10
unique row lengths before: [10]


,0,1,2,3,4,5,6,7,8,9
0,,Year Ended,Year Ended,One Year Change,One Year Change,"Year Ended December 31, 2017",One Year Change,One Year Change,Two Year Change,Two Year Change
1,,"December 31, 2015","December 31, 2016",$,%,"Year Ended December 31, 2017",$,%,$,%
2,NOI,"$1,175,806","$1,208,860","$33,054",3%,"$967,084","$(241,776)",-20%,"$(208,722)",-18%
3,Non-cash NOI attributable to same store proper...,"(48,890)","(38,899)","9,991",-20%,"(28,602)","10,297",-26%,"20,288",-41%
4,NOI attributable to non same store properties (2),"(498,131)","(574,049)","(75,918)",15%,"(333,279)","240,770",-42%,"164,852",-33%
5,SSNOI (1),"$628,785","$595,912","$(32,873)",-5%,"$605,203","$9,291",2%,"$(23,582)",-4%


### Batch Test All Complex Samples

In [13]:
parsed_tables = []
summary_records = []

for i, record in enumerate(tqdm(complex_samples.to_dict("records"), desc="Parsing complex samples")):
    imgid = record["imgid"]
    html = record["html_table"]

    result = robust_html_to_dataframe(html)

    parsed_tables.append({
        "imgid": imgid,
        "df": result["df"],
        "status": result["status"]
    })

    summary_records.append({
        "imgid": imgid,
        "max_colspan": record.get("max_colspan"),
        "max_rowspan": record.get("max_rowspan"),
        "status": result["status"],
        "error_type": result["error_type"],
        "error_message": result["error_message"],
        "normalized": result["normalized"],
        "min_cols_before": result["min_cols_before"],
        "max_cols_before": result["max_cols_before"],
        "unique_lengths_before": str(result["unique_lengths_before"]),
        "num_rows": result["num_rows"],
        "num_cols": result["num_cols"]
    })

df_parse_summary = pd.DataFrame(summary_records)

df_parse_summary.head()

Parsing complex samples:   0%|          | 0/1510 [00:00<?, ?it/s]

,imgid,max_colspan,max_rowspan,status,error_type,error_message,normalized,min_cols_before,max_cols_before,unique_lengths_before,num_rows,num_cols
0,fintabnet_000006,2,2,success_clean,,,False,10,10,[10],6,10
1,fintabnet_000008,2,2,success_clean,,,False,4,4,[4],12,4
2,fintabnet_000012,2,2,success_clean,,,False,10,10,[10],6,10
3,fintabnet_000026,2,2,success_clean,,,False,4,4,[4],12,4
4,fintabnet_000028,2,2,success_clean,,,False,10,10,[10],9,10


### Check Parsing Status

In [14]:
df_parse_summary["status"].value_counts()

,count
status,
success_clean,1495
success_fixed_misalignment,15


In [15]:
df_parse_summary[df_parse_summary["status"] == "failed"].head()

,imgid,max_colspan,max_rowspan,status,error_type,error_message,normalized,min_cols_before,max_cols_before,unique_lengths_before,num_rows,num_cols


In [16]:
df_parse_summary[df_parse_summary["normalized"] == True].head()

,imgid,max_colspan,max_rowspan,status,error_type,error_message,normalized,min_cols_before,max_cols_before,unique_lengths_before,num_rows,num_cols
8,fintabnet_000305,2,6,success_fixed_misalignment,,,True,4,5,"[4, 5]",7,5
27,fintabnet_002109,2,4,success_fixed_misalignment,,,True,5,7,"[5, 6, 7]",9,7
104,fintabnet_003457,3,3,success_fixed_misalignment,,,True,9,10,"[9, 10]",10,10
288,fintabnet_013610,2,3,success_fixed_misalignment,,,True,8,9,"[8, 9]",8,9
688,fintabnet_029880,4,6,success_fixed_misalignment,,,True,4,5,"[4, 5]",16,5


### Save Parsing Summary

In [17]:
summary_path = os.path.join(OUTPUT_DIR, "week3_complex_parse_summary.csv")

df_parse_summary.to_csv(summary_path, index=False)

print("Saved summary to:", summary_path)

Saved summary to: /content/drive/MyDrive/cv project/sprint3_final_outputs/sprint3_complex_parse_summary.csv


### Save parsed sample tables + original images

In [24]:
sample_output_dir = os.path.join(OUTPUT_DIR, "parsed_sample_tables")
os.makedirs(sample_output_dir, exist_ok=True)

saved_count = 0

for item in parsed_tables[11:21]:
    imgid = item["imgid"]
    df = item["df"]

    if df is not None:
        output_path = os.path.join(sample_output_dir, f"{imgid}_parsed.csv")
        df.to_csv(output_path, index=False)
        saved_count += 1

print("Saved parsed tables:", saved_count)
print("Output folder:", sample_output_dir)

Saved parsed tables: 10
Output folder: /content/drive/MyDrive/cv project/sprint3_final_outputs/parsed_sample_tables


In [28]:
import os
import shutil

base_path = "/content/drive/MyDrive/cv project/sprint3_final_outputs/week3_parsed_sample_tables"
os.makedirs(base_path, exist_ok=True)

# Original image folder
IMAGE_DIR = "/content/drive/MyDrive/cv project/hierarchical_tables_v1/data/processed/images"

# Save first 10 parsed samples
for i, item in enumerate(parsed_tables[:21]):
    imgid = item["imgid"]
    df = item["df"]

    # Create subfolder named by imgid
    sample_dir = os.path.join(base_path, imgid)
    os.makedirs(sample_dir, exist_ok=True)

    # 1. Save parsed DataFrame
    if df is not None:
        csv_path = os.path.join(sample_dir, f"{imgid}_parsed_table.csv")
        df.to_csv(csv_path, index=False)

    # 2. Save original image
    src_img_path = os.path.join(IMAGE_DIR, imgid + ".png")
    dst_img_path = os.path.join(sample_dir, f"{imgid}_original.png")

    try:
        shutil.copy(src_img_path, dst_img_path)
    except Exception as e:
        print(f"⚠️ Image copy failed for {imgid}: {e}")

print("Parsed tables and original images saved to:", base_path)

Parsed tables and original images saved to: /content/drive/MyDrive/cv project/sprint3_final_outputs/parsed_sample_tables


### View One Parsed Result

In [26]:
index = 10

print("imgid:", parsed_tables[index]["imgid"])
print("status:", parsed_tables[index]["status"])

display(parsed_tables[index]["df"])

imgid: fintabnet_000733
status: success_clean


,0,1,2,3,4,5,6
0,,,"As Restated December 30,2017",,As Restated,As Restated,
1,,"December 29,2018","As Restated December 30,2017",% Change,"December 30,2017","December 31,2016",% Change
2,,(in millions),(in millions),,(in millions),(in millions),
3,Net sales,"$26,268","$26,076",0.7%,"$26,076","$26,300",(0.9)%
4,Organic Net Sales (a),"26,105","25,876",0.9%,"25,963","26,188",(0.9)%


# Conclusion

Compared to Week 2, Week 3 upgrades the parser from a proof-of-concept to a robust, production-ready system with improved reliability, scalability, and interpretability. The following are upgrades:

- Finalized a unified parsing engine (robust_html_to_dataframe) to replace multiple experimental versions
- Introduced structured error handling for cases such as missing tables, empty tables, and parsing failures
- Added misalignment detection by checking row length consistency
- Applied conditional normalization to automatically fix misaligned rows only when needed
- Enabled large-scale batch processing and evaluation across complex samples
- Standardized outputs, including structured summary CSV and parsed table results for downstream use
- Clearly defined the boundary between parsing logic and HTML semantic issues to avoid unnecessary post-processing

In [43]:
# save to html
import json
import os

ipynb_path = "/content/drive/MyDrive/cv project/week3_final html-df parsing engine.ipynb"
clean_path = "/content/drive/MyDrive/cv project/week3_final_clean_for_html.ipynb"

with open(ipynb_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# remove widget metadata
nb.get("metadata", {}).pop("widgets", None)

# remove widget outputs only
for cell in nb.get("cells", []):
    if "outputs" in cell:
        new_outputs = []
        for output in cell["outputs"]:
            data = output.get("data", {})
            data.pop("application/vnd.jupyter.widget-view+json", None)
            data.pop("application/vnd.jupyter.widget-state+json", None)

            if data or output.get("text") or output.get("ename"):
                output["data"] = data
                new_outputs.append(output)

        cell["outputs"] = new_outputs

with open(clean_path, "w", encoding="utf-8") as f:
    json.dump(nb, f)

print("Saved cleaned notebook to:", clean_path)

Saved cleaned notebook to: /content/drive/MyDrive/cv project/week3_final_clean_for_html.ipynb


In [44]:
!jupyter nbconvert --to html --template classic "/content/drive/MyDrive/cv project/week3_final_clean_for_html.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/cv project/week3_final_clean_for_html.ipynb to html
[NbConvertApp] ERROR | Notebook JSON is invalid: Additional properties are not allowed ('data' was unexpected)

Failed validating 'additionalProperties' in stream:

On instance['cells'][7]['outputs'][0]:
{'data': {},
 'name': 'stdout',
 'output_type': 'stream',
 'text': 'Mounted at /content/drive\n'}
[NbConvertApp] Writing 375068 bytes to /content/drive/MyDrive/cv project/week3_final_clean_for_html.html
